# Decoding strategies, live

Companion notebook to **Lecture 13/14 — Decoding strategies**. Every strategy from the lecture, run on
the *same model and prompt*, so the differences are something you watch rather than take on faith.

- **Model:** `gpt2` (124M) — small, CPU-friendly, the same model the
  [Hugging Face blog](https://huggingface.co/blog/how-to-generate) uses.
- **Prompt:** *"I enjoy walking with my cute dog"* (the blog's prompt, so you can compare).
- **Requirements:** `pip install torch transformers`.

## 0. Setup

In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()   # quiet load

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id   # silence the pad warning

PROMPT = "I enjoy walking with my cute dog"
inputs = tokenizer(PROMPT, return_tensors="pt")

print(inputs)

def show(name, output_ids):
    print(f"[{name}]\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}\n")

print("gpt2 loaded on", model.device, "| vocab size", f"{model.config.vocab_size:,}")

{'input_ids': tensor([[   40,  2883,  6155,   351,   616, 13779,  3290]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
gpt2 loaded on cpu | vocab size 50,257


## 1. One decoding step is a distribution

The model does not emit a token — it emits a **score for every token in the vocabulary**. Softmax turns
those scores into a probability distribution; the decoding strategy is just *how you pick from it*.

In [3]:
with torch.no_grad():
    print(f"model output shape: {model(**inputs).logits.shape}")

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]        # next-token scores over the whole vocab
probs = F.softmax(logits, dim=-1)

print("Top 10 next tokens after the prompt:")
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"  {p.item():6.2%}  {tokenizer.decode([i])!r}")

top50 = torch.topk(probs, 50).values.sum().item()
print(f"\nvocabulary            : {probs.numel():,} tokens")
print(f"mass in the top 50    : {top50:.1%}")
print(f"mass in the other {probs.numel()-50:,}: {1-top50:.1%}   <- the long tail (lecture \u00a75)")

model output shape: torch.Size([1, 7, 50257])
Top 10 next tokens after the prompt:
  21.84%  ','
  15.40%  '.'
  13.85%  ' and'
   3.34%  ',"'
   2.58%  ' in'
   1.59%  ' on'
   1.44%  ' ('
   1.43%  '."'
   1.27%  ' because'
   1.24%  ' but'

vocabulary            : 50,257 tokens
mass in the top 50    : 78.8%
mass in the other 50,207: 21.2%   <- the long tail (lecture §5)


## 2. Greedy — and the repetition trap

`do_sample=False`: take the argmax every step. Watch it fall into the degenerate loop the lecture
warns about.

In [4]:
show("greedy", model.generate(**inputs, max_new_tokens=60, do_sample=False))

[greedy]
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk with my dog.

I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever



## 3. Beam search

Keep the `k` best partial sequences at each step. Higher sequence probability — but for open-ended
text it reads flat, and still loops. `no_repeat_ngram_size` bans repeated n-grams as a blunt patch.

In [5]:
show("beam k=5", model.generate(**inputs, max_new_tokens=60, num_beams=5, early_stopping=True))
show("beam k=5 + no_repeat_ngram_size=2",
     model.generate(**inputs, max_new_tokens=60, num_beams=5,
                    early_stopping=True, no_repeat_ngram_size=2))

[beam k=5]
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again.

I'm not sure

[beam k=5 + no_repeat_ngram_size=2]
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's time for me to take a step back and think about what I want to do next. I've always wanted to be



## 4. Sampling and temperature

Draw the next token at random, weighted by the distribution. **Temperature** reshapes it first:
low `T` sharpens toward the top token, high `T` flattens toward chaos.

In [5]:
for T in (0.7, 1.0, 1.5):
    torch.manual_seed(42)
    show(f"sampling T={T}",
         model.generate(**inputs, max_new_tokens=60, do_sample=True,
                        temperature=T, top_k=0, top_p=1.0))

[sampling T=0.7]
I enjoy walking with my cute dog, and I was delighted to have him."

The dog, also known as Bear, was born in March 2010. He was adopted from the Thomas and Catherine Salisbury Animal Shelter in Parkville, S.C., according to the his father, Camille Brundage.

"



[sampling T=1.0]
I enjoy walking with my cute dog, Daddy," a barter in the popular underground Twitter service Tweston tells Gladstone. "Like it or not, I've always wanted to dogfeed and cooperate with several others. Being an American cost me quite a bit, but as his father and mother have known each other from birth, I



[sampling T=1.5]
I enjoy walking with my cute dog Juno Daddy," a barter in Highlights will abound: whoever is closest will hang out talking weeeeeee farewell Yamato Midnight act Takeshi Itu – Thomas American loneliness Sal Meielve Dalou CR Alexandria cost cmempp North bands smyr hist await shaken concern Bruant allocations MurderSec instance Sugar



### Under the hood: what temperature does to the *logits*

No model needed — temperature is `softmax(logits / T)`. Reproduce the lecture's table on the real
next-token distribution: the same tokens, re-weighted.

In [6]:
print("token".ljust(14) + "".join(f"T={T:<8}" for T in (0.5, 1.0, 2.0)))
for i in torch.topk(probs, 5).indices:
    row = repr(tokenizer.decode([i])).ljust(14)
    for T in (0.5, 1.0, 2.0):
        row += f"{F.softmax(logits / T, dim=-1)[i].item():<10.2%}"
    print(row)

token         T=0.5     T=1.0     T=2.0     
','           50.59%    21.84%    1.08%     
'.'           25.15%    15.40%    0.90%     
' and'        20.36%    13.85%    0.86%     
',"'          1.19%     3.34%     0.42%     
' in'         0.71%     2.58%     0.37%     


## 5. Top-k and nucleus (top-p)

Cut the distribution down to a trusted set, renormalise, sample only inside it. Top-k keeps a fixed
count; top-p keeps the smallest set reaching cumulative probability `p`.

In [7]:
torch.manual_seed(42)
show("top-k = 50", model.generate(**inputs, max_new_tokens=60, do_sample=True, top_k=50))
torch.manual_seed(42)
show("top-p = 0.92", model.generate(**inputs, max_new_tokens=60, do_sample=True, top_k=0, top_p=0.92))

[top-k = 50]
I enjoy walking with my cute dog, which is a little unusual in this part of our family. He's a friendly, calm kind of dog, and I've always wanted to have him around, and I always wanted to go with him when he comes to pick me up from his crate or his seat. I want to give him



[top-p = 0.92]
I enjoy walking with my cute dog, Daddy," a barter in the popular underground Twitter service Twitchell agreed.

Like it or not, "It's been a while," American Indian Salwa Donna Dal made me laugh at how silly men as bands were simply hisers and slugs after all, but Dal became a



### Under the hood: how big is the surviving set?

The lecture's key claim: top-k keeps 50 no matter what, while top-p **adapts** — few tokens on a
peaked step, many on a flat one. Measure it on two prompts.

In [8]:
def nucleus_size(logits, p):
    s, _ = torch.sort(F.softmax(logits, dim=-1), descending=True)
    return int((torch.cumsum(s, dim=-1) < p).sum().item()) + 1   # +1: the token that crosses p

for prompt in ["The United States of",   # peaked — gpt2 is 96% sure the next token is " America"
               "My favourite food is"]:   # flat   — dozens of reasonable continuations
    ids = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        lg = model(**ids).logits[0, -1]
    print(f"{prompt!r}")
    print(f"   nucleus (p=0.9) size = {nucleus_size(lg, 0.9):>5}   (top-k=50 keeps 50 regardless)\n")

'The United States of'
   nucleus (p=0.9) size =     1   (top-k=50 keeps 50 regardless)

'My favourite food is'
   nucleus (p=0.9) size =  1913   (top-k=50 keeps 50 regardless)



## 6. Side by side

Same model, same prompt, every strategy — the comparison the lecture pointed here for.

In [9]:
strategies = {
    "greedy":       dict(do_sample=False),
    "beam k=5":     dict(num_beams=5, early_stopping=True),
    "sampling T=1": dict(do_sample=True, temperature=1.0, top_k=0, top_p=1.0),
    "top-k=50":     dict(do_sample=True, top_k=50),
    "top-p=0.92":   dict(do_sample=True, top_k=0, top_p=0.92),
}
for name, kw in strategies.items():
    torch.manual_seed(42)
    out = model.generate(**inputs, max_new_tokens=40, **kw)
    cont = tokenizer.decode(out[0], skip_special_tokens=True)[len(PROMPT):].strip()
    print(f"[{name:<12}] \u2026{cont}\n")

[greedy      ] …, but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk with my dog.

I'm not sure



[beam k=5    ] …, but I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again. I'm not sure



[sampling T=1] …, Daddy," a barter in the popular underground Twitter service Tweston tells Gladstone. "Like it or not, I've always wanted to dogfeed and cooperate with several others. Being an American



[top-k=50    ] …, which is a little unusual in this part of our family. He's a friendly, calm kind of dog, and I've always wanted to have him around, and I always wanted to go with



[top-p=0.92  ] …, Daddy," a barter in the popular underground Twitter service Twitchell agreed.

Like it or not, "It's been a while," American Indian Salwa Donna Dal made me laugh



## Exercises

1. **Break greedy.** Find the shortest prompt for which greedy loops within 40 tokens. What property
   of the prompt causes it?
2. **Temperature at the extremes.** Generate at `T=0.01` and `T=3.0`. Explain each in terms of the
   ratio `P_i / P_j = exp((l_i - l_j)/T)` from the lecture.
3. **Reproduce top-p yourself.** Write a `sample_top_p(logits, p)` that truncates, renormalises, and
   draws — then check its output distribution matches `generate(..., top_p=p)` over many draws.
4. **Peaked vs flat, quantified.** Extend §5's measurement to 20 prompts and plot nucleus size against
   the entropy of the next-token distribution.

See **Lecture 13/14** for the theory behind each strategy.